# Data Exploration - UNSW-NB15

**Objective:** Understand dataset structure, class distribution, feature types, and correlations.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

plt.style.use('ggplot')
%matplotlib inline

In [ ]:
from src.data import DataLoader, DataCleaner

loader = DataLoader(config)
train_df, test_df = loader.load_raw()
df = loader.merge_partitions(train_df, test_df)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
df.head()

In [ ]:
# Class distribution
label_counts = df['label'].value_counts()
print(f"Label distribution:\n{label_counts}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
label_counts.plot(kind='bar', ax=axes[0], title='Binary Label Distribution')
axes[0].set_xticklabels(['Benign', 'Attack'], rotation=0)

if 'attack_cat' in df.columns:
    cat_counts = df['attack_cat'].value_counts()
    cat_counts.plot(kind='bar', ax=axes[1], title='Attack Category Distribution')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150)

In [ ]:
# Feature types
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Categorical columns: {categorical_cols}")

In [ ]:
# Missing values
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print(f"Columns with missing values:\n{missing}")
else:
    print("No missing values found")

In [ ]:
# Correlation with target
correlations = df[numeric_cols].corrwith(df['label']).abs().sort_values(ascending=False)
print("Top 15 features correlated with label:")
print(correlations.head(15))

In [ ]:
# High-correlation pairs (redundant features)
corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, row, upper.loc[row, col]) 
             for col in upper.columns 
             for row in upper.index 
             if upper.loc[row, col] > 0.95]
print(f"Highly correlated feature pairs (>0.95): {len(high_corr)}")
for pair in high_corr[:10]:
    print(f"  {pair[0]} ~ {pair[1]}: {pair[2]:.3f}")

In [ ]:
# Partition sizes
print(f"Original train size: {len(train_df)}")
print(f"Original test size: {len(test_df)}")
print(f"Combined size: {len(df)}")